# HotWAY: Neural TSP Training via Attention Model (Kool et al., ICLR 2019)
### Accelerated Training Pipeline for Time-Dependent Urban Logistics (TP. Hồ Chí Minh)

**Hardware Target:** Kaggle GPU (NVIDIA T4 x2, 16GB VRAM)
**Objective:** Train an end-to-end Attention Model to solve dynamic TSP on 20-50 nodes with empirical congestion factor $C(t)$, exporting weights for microsecond CPU inference in FastAPI.

In [ ]:
import math
import time
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Executing training on device: {device}")

In [ ]:
# Empirical congestion multiplier function C(t) for TP.HCM
def get_congestion_factor(hour_of_day: float) -> float:
    h = hour_of_day % 24.0
    if 22.0 <= h or h < 6.0: return 1.00
    elif 6.0 <= h < 7.0: return 1.15
    elif 7.0 <= h < 8.5: return 2.10  # Morning rush
    elif 8.5 <= h < 11.5: return 1.25
    elif 11.5 <= h < 13.5: return 1.35 # Lunch rush
    elif 13.5 <= h < 16.5: return 1.20
    elif 16.5 <= h < 19.0: return 2.25 # Evening rush
    elif 19.0 <= h < 22.0: return 1.30
    return 1.00

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, n_heads=8, input_dim=128, embed_dim=128):
        super().__init__()
        self.n_heads = n_heads
        self.embed_dim = embed_dim
        self.head_dim = embed_dim // n_heads
        self.w_q = nn.Linear(input_dim, embed_dim, bias=False)
        self.w_k = nn.Linear(input_dim, embed_dim, bias=False)
        self.w_v = nn.Linear(input_dim, embed_dim, bias=False)
        self.w_out = nn.Linear(embed_dim, embed_dim, bias=False)

    def forward(self, q, k=None, v=None, mask=None):
        if k is None: k = q
        if v is None: v = q
        b, q_len = q.size(0), q.size(1)
        k_len = k.size(1)
        q_s = self.w_q(q).view(b, q_len, self.n_heads, self.head_dim).transpose(1, 2)
        k_s = self.w_k(k).view(b, k_len, self.n_heads, self.head_dim).transpose(1, 2)
        v_s = self.w_v(v).view(b, k_len, self.n_heads, self.head_dim).transpose(1, 2)
        scores = torch.matmul(q_s, k_s.transpose(-2, -1)) / math.sqrt(self.head_dim)
        if mask is not None:
            scores = scores.masked_fill(mask == 1, -1e9)
        attn = F.softmax(scores, dim=-1)
        context = torch.matmul(attn, v_s).transpose(1, 2).contiguous().view(b, q_len, self.embed_dim)
        return self.w_out(context)

class EncoderLayer(nn.Module):
    def __init__(self, embed_dim=128, n_heads=8, ffn_dim=512):
        super().__init__()
        self.mha = MultiHeadAttention(n_heads=n_heads, input_dim=embed_dim, embed_dim=embed_dim)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.ffn = nn.Sequential(nn.Linear(embed_dim, ffn_dim), nn.ReLU(), nn.Linear(ffn_dim, embed_dim))
        self.norm2 = nn.LayerNorm(embed_dim)

    def forward(self, x):
        h = self.norm1(x + self.mha(x))
        return self.norm2(h + self.ffn(h))

class AttentionModel(nn.Module):
    def __init__(self, input_dim=3, embed_dim=128, n_heads=8, n_layers=3):
        super().__init__()
        self.init_embed = nn.Linear(input_dim, embed_dim)
        self.encoder_layers = nn.ModuleList([EncoderLayer(embed_dim, n_heads) for _ in range(n_layers)])
        self.context_proj = nn.Linear(embed_dim * 3 + 1, embed_dim)
        self.dec_mha = MultiHeadAttention(n_heads, embed_dim, embed_dim)
        self.w_q_ptr = nn.Linear(embed_dim, embed_dim, bias=False)
        self.w_k_ptr = nn.Linear(embed_dim, embed_dim, bias=False)
        self.embed_dim = embed_dim

    def forward(self, nodes, decode_type="greedy"):
        b, n, _ = nodes.size()
        h = self.init_embed(nodes)
        for layer in self.encoder_layers:
            h = layer(h)
        visited = torch.zeros(b, n, dtype=torch.uint8, device=nodes.device)
        curr = torch.zeros(b, dtype=torch.long, device=nodes.device)
        depot = torch.zeros(b, dtype=torch.long, device=nodes.device)
        visited[:, 0] = 1
        tours, log_probs = [curr], []
        t_ratio = torch.zeros(b, 1, device=nodes.device)
        b_idx = torch.arange(b, device=nodes.device)

        for step in range(1, n):
            ctx = torch.cat([h.mean(dim=1), h[b_idx, depot], h[b_idx, curr], t_ratio], dim=-1)
            q = self.context_proj(ctx).unsqueeze(1)
            q_mha = self.dec_mha(q, h, h, mask=visited.unsqueeze(1).unsqueeze(2))
            scores = torch.matmul(self.w_q_ptr(q_mha), self.w_k_ptr(h).transpose(-2, -1)).squeeze(1) / math.sqrt(self.embed_dim)
            logits = (10.0 * torch.tanh(scores)).masked_fill(visited == 1, -1e9)
            probs = F.softmax(logits, dim=-1)
            next_n = torch.argmax(probs, dim=-1) if decode_type == "greedy" else torch.multinomial(probs, 1).squeeze(-1)
            log_probs.append(F.log_softmax(logits, dim=-1).gather(1, next_n.unsqueeze(-1)).squeeze(-1))
            visited[b_idx, next_n] = 1
            curr = next_n
            tours.append(curr)
            t_ratio = t_ratio + (1.0 / n)

        return torch.stack(tours, dim=1), torch.stack(log_probs, dim=1).sum(dim=1)

In [ ]:
# REINFORCE Training Loop with Greedy Rollout Baseline
model = AttentionModel(input_dim=3, embed_dim=128, n_heads=8, n_layers=3).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.988)

print("Model initialized. Ready for training on Kaggle GPU!")

In [ ]:
# Export artifacts after training
torch.save(model.state_dict(), "attention_tsp50_hcmc.pt")
print("Saved weights to attention_tsp50_hcmc.pt for deployment!")